# Bab 10 — Studi Kasus dan Implementasi Statistical Machine Learning
## Klasifikasi Varietas Beras: Rice (Cammeo and Osmancik)
**Buku pemula · Notebook Google Colab v1.1 · Repositori pendamping buku · Python · 3 algoritma**

**Sumber asli:** Cinar & Koklu, *Rice (Cammeo and Osmancik)* (2019), UCI Machine Learning Repository, DOI [10.24432/C5MW4Z](https://doi.org/10.24432/C5MW4Z), lisensi **CC BY 4.0**. Dokumentasi: https://archive.ics.uci.edu/dataset/545/rice.

Dataset memuat 3.810 hasil pengukuran butir beras dari dua varietas, masing-masing memiliki **tujuh fitur morfologi**. Input berupa angka hasil ekstraksi citra, **bukan foto butir beras**.

**Tujuan:** membandingkan Logistic Regression, K-Nearest Neighbors, dan Decision Tree, serta memahami statistika deskriptif, pembagian data, validasi silang, dan confusion matrix.

**Protokol yang dikunci sebelum melihat testing:** audit data → hapus baris identik untuk mencegah pengulangan yang sama melintasi partisi → split *stratified* 80:20 (seed 42) → EDA training → CV stratified lima fold pada training → pilih model menurut rata-rata Macro-F1 CV → latih dan evaluasi semua model pada test satu kali. Tidak ada hyperparameter tuning.
**Versi kode:** 1.1 (penyelarasan nama tabel/gambar; protokol dan parameter tidak diubah).
**Catatan:** notebook ini dapat dijalankan tanpa GPU. Jika versi pustaka berbeda, hasil numerik mungkin sedikit berubah. Keluaran acuan tersedia di folder `results/`.


## 10.1 Pengenalan Implementasi Statistical Machine Learning

Eksperimen ini sengaja sederhana. Model bukan sekadar dinilai dari skor tertinggi; pembaca juga diajak memahami kesalahan klasifikasi dan mengapa standardisasi fitur diperlukan untuk model berbasis jarak.

In [ ]:
from pathlib import Path
import io, zipfile, json, platform, hashlib
from urllib.request import urlopen, Request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy.io import arff
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             precision_recall_fscore_support)

SEED = 42
DATASET_PAGE = 'https://archive.ics.uci.edu/dataset/545/rice'
DATA_URL = 'https://archive.ics.uci.edu/static/public/545/rice%2Bcammeo%2Band%2Bosmancik.zip'
FALLBACK_URL = 'https://archive.ics.uci.edu/static/public/545/rice+cammeo+and+osmancik.zip'
FEATURES = ['Area','Perimeter','Major_Axis_Length','Minor_Axis_Length',
            'Eccentricity','Convex_Area','Extent']
OUTPUT = Path.cwd() / 'Bab10_Rice_Results_v1_1'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__, '| pandas:', pd.__version__)
print('Output:', OUTPUT)
# Checkpoint sumber data acuan (ARFF asli yang digunakan menghasilkan tabel buku).
EXPECTED_SHA256 = '1af97883100c89de2ea2972f7a28d428f4f1c14711a61defc0b0569e9eb65665'


## 10.2 Pengenalan Dataset dan Statistik Deskriptif

Tujuh fitur resmi adalah `Area` (luas), `Perimeter` (keliling), `Major_Axis_Length` (panjang sumbu utama), `Minor_Axis_Length` (panjang sumbu minor), `Eccentricity` (eksentrisitas), `Convex_Area` (luas selubung cembung), dan `Extent` (rasio luas area terhadap *bounding box*). Target berupa varietas **Cammeo** atau **Osmancik**.

Notebook mengunduh arsip ZIP resmi UCI dan membaca berkas `.arff` di dalamnya. Jika koneksi Colab gagal, unduh arsip secara manual dari halaman UCI lalu unggah `rice+cammeo+and+osmancik.zip` **atau** `Rice_Cammeo_Osmancik.arff` ke folder `/content`.

In [ ]:
def locate_arff_bytes():
    # Prioritaskan unggahan manual agar notebook dapat dipakai tanpa koneksi ketika data telah tersedia.
    candidates = [Path.cwd()/'Rice_Cammeo_Osmancik.arff', Path('/content/Rice_Cammeo_Osmancik.arff'),
                  Path.cwd()/'rice+cammeo+and+osmancik.zip', Path('/content/rice+cammeo+and+osmancik.zip'),
                  Path.cwd()/'rice_uci_545.zip', Path('/content/rice_uci_545.zip')]
    def from_blob(blob):
        if blob[:2] == b'PK':
            with zipfile.ZipFile(io.BytesIO(blob)) as z:
                names = [n for n in z.namelist() if n.lower().endswith('.arff') and not n.endswith('/')]
                if not names:
                    raise ValueError('Tidak ditemukan berkas .arff di dalam arsip ZIP.')
                names.sort(key=lambda x: (not x.lower().endswith('rice_cammeo_osmancik.arff'),len(x)))
                return z.read(names[0])
        if b'@relation' in blob[:2500].lower() or b'@attribute' in blob[:2500].lower():
            return blob
        raise ValueError('Isi file bukan ARFF/ZIP dataset Rice yang diharapkan.')
    for p in candidates:
        if p.is_file():
            return from_blob(p.read_bytes()), str(p)
    errors=[]
    for url in [DATA_URL, FALLBACK_URL]:
        try:
            req=Request(url,headers={'User-Agent':'Mozilla/5.0 (Educational notebook)'})
            with urlopen(req,timeout=45) as r: blob=r.read()
            return from_blob(blob), url
        except Exception as exc:
            errors.append(str(exc))
    raise RuntimeError(
        'Data belum tersedia. Unduh arsip dari '+DATASET_PAGE+
        ', unggah ZIP atau Rice_Cammeo_Osmancik.arff ke /content, lalu jalankan ulang sel ini. '
        'Detail kegagalan download: '+' | '.join(errors)
    )

ARFF_BYTES, DATA_ORIGIN = locate_arff_bytes()
DATA_SHA256 = hashlib.sha256(ARFF_BYTES).hexdigest()
raw, _meta = arff.loadarff(io.StringIO(ARFF_BYTES.decode('utf-8-sig', errors='replace')))
df_raw = pd.DataFrame(raw)
# scipy.io.arff menyimpan string kategorikal sebagai bytes; konversi agar mudah dipahami.
for col in df_raw.select_dtypes(include=['object']).columns:
    df_raw[col] = df_raw[col].map(lambda v: v.decode('utf-8') if isinstance(v,bytes) else v)
# Normalisasi ejaan atribut, mempertahankan nilai numeriknya.
if set(FEATURES+['Class']) != set(df_raw.columns):
    raise ValueError(f'Kolom aktual: {list(df_raw.columns)}; diharapkan tujuh fitur dan Class.')
df_raw = df_raw[FEATURES+['Class']]
for f in FEATURES:
    df_raw[f] = pd.to_numeric(df_raw[f],errors='raise')
if len(df_raw)!=3810 or set(df_raw['Class'])!={'Cammeo','Osmancik'}:
    raise ValueError('Ukuran atau label berbeda dari dokumentasi resmi UCI; periksa file.')
print('Sumber:', DATA_ORIGIN)
print('SHA-256 ARFF:', DATA_SHA256)
print('Bentuk data mentah:', df_raw.shape)
display(df_raw.head())
if DATA_SHA256 != EXPECTED_SHA256:
    print('PERINGATAN: SHA-256 ARFF berbeda dari sumber data acuan buku. Periksa versi dataset sebelum membandingkan angka!')
else:
    print('SHA-256 cocok dengan data acuan buku.')


In [ ]:
missing=int(df_raw.isna().sum().sum())
duplicate=int(df_raw.duplicated().sum())
conflicts=int((df_raw.groupby(FEATURES,dropna=False)['Class'].nunique()>1).sum())
if missing or conflicts:
    raise ValueError(f'Audit data memerlukan pemeriksaan manual: missing={missing}, konflik label={conflicts}')
# Hanya duplikasi identik seluruh kolom yang dihapus.
df = df_raw.drop_duplicates().reset_index(drop=True)
audit = pd.DataFrame([
    ('Observasi awal', len(df_raw)), ('Fitur numerik',len(FEATURES)),
    ('Jumlah kelas target',df_raw['Class'].nunique()),
    ('Missing values',missing), ('Duplikasi identik',duplicate),
    ('Konflik label untuk fitur identik',conflicts),('Observasi setelah audit',len(df))
],columns=['Indikator','Nilai'])
display(audit)
audit.to_csv(OUTPUT/'Tabel_10_2_Audit_Dataset.csv',index=False)
print('Jumlah setiap kelas:\n', df['Class'].value_counts())
# Tabel 10.1 buku: definisi fitur mengacu pada metadata resmi UCI.
feature_definitions = [
 ('Area','Luas butir beras di dalam batas objek','piksel'),
 ('Perimeter','Keliling batas butir beras','piksel'),
 ('Major_Axis_Length','Panjang sumbu utama bentuk butir','satuan panjang pada citra'),
 ('Minor_Axis_Length','Panjang sumbu minor bentuk butir','satuan panjang pada citra'),
 ('Eccentricity','Ukuran kelonjongan elips yang mewakili butir','tanpa satuan'),
 ('Convex_Area','Luas selubung cembung butir','piksel'),
 ('Extent','Rasio luas butir terhadap kotak pembatas','tanpa satuan'),
 ('Class','Label target varietas: Cammeo atau Osmancik','kategori')
]
pd.DataFrame(feature_definitions,columns=['Variabel','Keterangan','Satuan atau skala']).to_csv(OUTPUT/'Tabel_10_1_Deskripsi_Fitur.csv',index=False)


## 10.3 Persiapan Data dan Pembagian Dataset

Lakukan split terstratifikasi **80% training dan 20% testing**. Jangan menstandardisasi seluruh dataset sebelum split, sebab informasi dari testing dapat bocor ke training. Statistik deskriptif dan dua grafik eksplorasi di bawah ini dihitung hanya dari training.

In [ ]:
X = df[FEATURES].copy()
y = df['Class'].copy()
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
split = pd.DataFrame([
    {'Subset':'Training','Observasi':len(X_train), **y_train.value_counts().to_dict()},
    {'Subset':'Testing','Observasi':len(X_test), **y_test.value_counts().to_dict()}
]).fillna(0)
split=pd.concat([split,pd.DataFrame([{'Subset':'Total','Observasi':len(df),
    **df['Class'].value_counts().to_dict()}])],ignore_index=True)
display(split)
split.to_csv(OUTPUT/'Tabel_10_3_Pembagian_Data.csv',index=False)
stats=X_train.describe().T.round(3)
display(stats)
stats.to_csv(OUTPUT/'Lampiran_Statistik_Deskriptif.csv')

In [ ]:
# Gambar 10.1 – kelas data latih
fig, ax=plt.subplots(figsize=(6.7,4.1))
counts=y_train.value_counts().sort_index()
ax.bar(counts.index,counts.values)
for i,v in enumerate(counts.values): ax.text(i,v+10,str(v),ha='center')
ax.set(title='Distribusi Varietas Beras pada Data Latih',xlabel='Varietas',ylabel='Jumlah observasi')
fig.tight_layout();fig.savefig(OUTPUT/'Gambar_10_1_Distribusi_Kelas.png',dpi=170);plt.show()

# Gambar 10.2 – contoh hubungan dua fitur yang mudah ditafsirkan
fig,ax=plt.subplots(figsize=(7,4.6))
for cls in sorted(y_train.unique()):
    z=X_train.loc[y_train==cls]
    ax.scatter(z['Major_Axis_Length'],z['Area'],s=11,alpha=.55,label=cls)
ax.set(xlabel='Major Axis Length',ylabel='Area',title='Hubungan Panjang Sumbu Utama dan Luas Butir Beras')
ax.legend();fig.tight_layout();fig.savefig(OUTPUT/'Gambar_10_2_Scatter_Training.png',dpi=170);plt.show()
# Gambar 10.3 dalam naskah merupakan diagram alur editorial,
# bukan grafik data; file pendampingnya ada di direktori figures/ repositori.


## 10.4 Implementasi Algoritma Machine Learning dengan Python

Tiga model: **Logistic Regression** (parametrik), **KNN dengan k=5** (berbasis jarak), dan **Decision Tree dengan kedalaman maksimum 4** (berbasis aturan). StandardScaler berada **di dalam Pipeline** Logistic Regression dan KNN. Validasi silang hanya melibatkan training: Stratified 5-Fold (acak dengan seed 42).

**Aturan pemilihan:** rerata Macro-F1 CV tertinggi. Jika sama persis, gunakan urutan Logistic Regression, KNN, Decision Tree. Performa test tidak digunakan untuk memilih model.

In [ ]:
models={
 'Logistic Regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=SEED)),
 'K-Nearest Neighbors':make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5)),
 'Decision Tree':DecisionTreeClassifier(max_depth=4,random_state=SEED),
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
cv_rows=[]
for name,model in models.items():
    r=cross_validate(model,X_train,y_train,cv=cv,
                     scoring={'accuracy':'accuracy','macro_f1':'f1_macro'},n_jobs=1)
    cv_rows.append({'Model':name,
        'CV_Accuracy_Mean':r['test_accuracy'].mean(),
        'CV_Accuracy_SD':r['test_accuracy'].std(ddof=1),
        'CV_Macro_F1_Mean':r['test_macro_f1'].mean(),
        'CV_Macro_F1_SD':r['test_macro_f1'].std(ddof=1)})
cv_result=pd.DataFrame(cv_rows).sort_values('CV_Macro_F1_Mean',ascending=False,kind='stable').reset_index(drop=True)
selected=cv_result.loc[0,'Model']
print('Model terpilih dari CV pada data training:',selected)
display(cv_result.round(4))
cv_result.to_csv(OUTPUT/'Tabel_10_4_Hasil_Validasi_Silang.csv',index=False)

In [ ]:
fig,ax=plt.subplots(figsize=(8.1,4.3))
ax.bar(cv_result['Model'],cv_result['CV_Macro_F1_Mean'],yerr=cv_result['CV_Macro_F1_SD'],capsize=4)
ax.set(ylim=(0,1.02),title='Rata-rata Macro-F1 Validasi Silang (Training)',ylabel='Macro-F1',xlabel='Algoritma')
ax.tick_params(axis='x',labelrotation=12)
fig.tight_layout();fig.savefig(OUTPUT/'Gambar_10_4_Validasi_Silang.png',dpi=170);plt.show()

## 10.5 Evaluasi dan Perbandingan Kinerja Model

Latih kembali ketiga model pada seluruh training; evaluasi **sekali** pada testing. Metrik yang digunakan adalah *accuracy*, *precision macro*, *recall macro*, dan *Macro-F1*. Karena klasifikasi memiliki dua kelas, Macro-F1 menghitung rata-rata F1 kedua kelas secara seimbang. Confusion matrix memberi contoh konkret jenis kesalahan.

In [ ]:
test_rows=[]
predictions={'actual':y_test.to_numpy()}
for name,model in models.items():
    model.fit(X_train,y_train)
    p=model.predict(X_test)
    predictions[name]=p
    test_rows.append({'Model':name,
        'Accuracy':accuracy_score(y_test,p),
        'Precision_Macro':precision_score(y_test,p,average='macro',zero_division=0),
        'Recall_Macro':recall_score(y_test,p,average='macro',zero_division=0),
        'Macro_F1':f1_score(y_test,p,average='macro',zero_division=0),
        'Jumlah_Salah':int((p!=y_test.to_numpy()).sum())})
test_result=pd.DataFrame(test_rows)
display(test_result.round(4))
test_result.to_csv(OUTPUT/'Tabel_10_5_Perbandingan_Model_Test.csv',index=False)
pd.DataFrame(predictions).to_csv(OUTPUT/'Prediksi_Pengujian.csv',index=False)
selected_pred=predictions[selected]
print('Model terpilih:',selected)
print(classification_report(y_test,selected_pred,zero_division=0))

In [ ]:
labels=['Cammeo','Osmancik']
cm=confusion_matrix(y_test,selected_pred,labels=labels)
fig,ax=plt.subplots(figsize=(6.6,5.2))
ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False,values_format='d')
ax.set_title(f'Confusion Matrix — {selected}')
fig.tight_layout();fig.savefig(OUTPUT/'Gambar_10_5_Confusion_Matrix.png',dpi=170);plt.show()
errors = pd.DataFrame({'Actual':y_test.to_numpy(),'Prediksi':selected_pred})
wrong=errors.loc[errors['Actual'] != errors['Prediksi']]
print('Jumlah observasi salah klasifikasi:',len(wrong),'dari',len(errors))
# Tabel 10.6 buku: metrik per kelas untuk model terpilih.
per_precision,per_recall,per_f1,per_support=precision_recall_fscore_support(
    y_test, selected_pred,labels=labels,zero_division=0)
per_class=pd.DataFrame({'Kelas':labels,'Precision':per_precision,'Recall':per_recall,
                        'F1':per_f1,'Support':per_support})
per_class=pd.concat([per_class,pd.DataFrame([{'Kelas':'Rata-rata macro',
    'Precision':float(np.mean(per_precision)),'Recall':float(np.mean(per_recall)),
    'F1':float(np.mean(per_f1)),'Support':int(np.sum(per_support))}])],ignore_index=True)
per_class.to_csv(OUTPUT/'Tabel_10_6_Metrik_Per_Kelas.csv',index=False)
display(per_class.round(4))


## 10.6 Interpretasi Hasil, Praktik Baik, dan Kesimpulan

Perhatikan bahwa metrik pengujian berasal dari **satu pembagian data**; nilainya dapat berubah pada data atau pengacakan berbeda. Grafik dan tabel hanya berlaku untuk dataset Rice ini, bukan bukti bahwa model otomatis akan bekerja sama baik pada semua varietas beras.

**Pelajaran utama:** (a) lakukan split sebelum scaling, (b) lakukan CV hanya pada training, (c) bandingkan lebih dari satu metrik, (d) gunakan confusion matrix untuk menelaah kesalahan, dan (e) dokumentasikan versi paket dan sumber dataset.

Untuk membandingkan dengan Banknote atau Iris, ingat bahwa dataset dan sampel testing berbeda; membandingkan persentase akurasi secara langsung **bukan** pengujian superioritas algoritma lintas dataset.

In [ ]:
summary={
 'dataset':'UCI Rice (Cammeo and Osmancik)',
 'dataset_doi':'10.24432/C5MW4Z',
 'data_origin':DATA_ORIGIN,
 'sha256_arff':DATA_SHA256,
 'raw_rows':int(len(df_raw)),
 'duplicate_rows_removed':int(duplicate),
 'rows_analyzed':int(len(df)),
 'train_rows':int(len(X_train)),
 'test_rows':int(len(X_test)),
 'selected_model_from_training_cv':selected,
 'random_state':SEED,
 'test_evaluated_once':True,
 'versions':{'python':platform.python_version(),'numpy':np.__version__,
             'pandas':pd.__version__,'scikit_learn':sklearn.__version__},
}
(OUTPUT/'Metadata_Eksperimen.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
(OUTPUT/'README_Hasil.txt').write_text('Eksperimen UCI Rice untuk Bab 10. Sumber: '+DATASET_PAGE+
    '\nTabel dan gambar dari hasil eksekusi pada dataset asli.\n',encoding='utf-8')
archive=Path.cwd()/'Bab10_Rice_Results_v1_1.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT.iterdir()): z.write(f,arcname=f.name)
print('Arsip output siap:',archive)
print('Berkas keluaran:',len(list(OUTPUT.iterdir())))
# Bagi pengguna Google Colab, tampilkan link unduhan bila tersedia.
try:
    from google.colab import files
    # Klik file di panel Files untuk download; baris berikut opsional jika ingin otomatis:
    # files.download(str(archive))
except ImportError:
    pass
# Verifikasi cepat angka utama (tidak mengubah hasil eksperimen).
assert len(df_raw)==3810 and len(X_train)==3048 and len(X_test)==762
if DATA_SHA256 == EXPECTED_SHA256:
    assert selected=='Logistic Regression'
else:
    print('Dataset berbeda dari referensi; pemilihan model dan angka dapat berbeda.')
print('Ringkasan akhir: accuracy =',round(accuracy_score(y_test,selected_pred),4),
      '; salah =',int((selected_pred!=y_test.to_numpy()).sum()))
